In [0]:
%sh
pip install "protobuf>=5.26.1,<6.0dev" "dbt-databricks" --quiet

# Create project directory structure
mkdir -p globalmart_dbt/models/staging
mkdir -p globalmart_dbt/models/marts
mkdir -p globalmart_dbt/snapshots
mkdir -p globalmart_dbt/macros
mkdir -p globalmart_dbt/tests

In [0]:
%sh
pip install "protobuf>=5.26.1,<6.0dev" "dbt-databricks" --quiet

In [0]:
import os

os.makedirs(os.path.expanduser("~/.dbt"), exist_ok=True)

profiles_yaml = """
globalmart_dbt:
  target: dev
  outputs:
    dev:
      type: databricks
      catalog: globalmart
      schema: bronze_dev
      host: "dbc-73a59226-9cbc.cloud.databricks.com"
      http_path: "/sql/1.0/warehouses/bcfd52b943980648"
      token: "dc30e61be4a568eca596c9139797ddb4af865d49cd1e1ab83e6f90f98ef7d"
      threads: 4

    prod:
      type: databricks
      catalog: globalmart
      schema: bronze
      host: "dbc-73a59226-9cbc.cloud.databricks.com"
      http_path: "/sql/1.0/warehouses/bcfd52b943980648"
      token: "c30e61be4a568eca596c9139797ddb4af865d49cd1e1ab83e6f90f98ef7d"
      threads: 8
"""

with open(os.path.expanduser("~/.dbt/profiles.yml"), "w") as f:
    f.write(profiles_yaml.strip())

print("✅ profiles.yml created!")

In [0]:
import os

os.makedirs(os.path.expanduser("~/.dbt"), exist_ok=True)

# Using auth_type: oauth leverages your active Databricks notebook session automatically!
profiles_yaml = """
globalmart_dbt:
  target: dev
  outputs:
    dev:
      type: databricks
      catalog: globalmart
      schema: bronze_dev
      host: "dbc-73a59226-9cbc.cloud.databricks.com"
      http_path: "/sql/1.0/warehouses/bcfd52b943980648"
      auth_type: "oauth"
      threads: 4

    prod:
      type: databricks
      catalog: globalmart
      schema: bronze
      host: "dbc-73a59226-9cbc.cloud.databricks.com"
      http_path: "/sql/1.0/warehouses/bcfd52b943980648"
      auth_type: "oauth"
      threads: 8
"""

with open(os.path.expanduser("~/.dbt/profiles.yml"), "w") as f:
    f.write(profiles_yaml.strip())

print("✅ Updated profiles.yml with OAuth authentication!")

In [0]:
import os

os.makedirs(os.path.expanduser("~/.dbt"), exist_ok=True)

profiles_yaml = """
globalmart_dbt:
  target: dev
  outputs:
    dev:
      type: databricks
      catalog: globalmart
      schema: bronze_dev
      host: "dbc-73a59226-9cbc.cloud.databricks.com"
      http_path: "/sql/1.0/warehouses/bcfd52b943980648"
      auth_type: "databricks-cli"
      threads: 4

    prod:
      type: databricks
      catalog: globalmart
      schema: bronze
      host: "dbc-73a59226-9cbc.cloud.databricks.com"
      http_path: "/sql/1.0/warehouses/bcfd52b943980648"
      auth_type: "databricks-cli"
      threads: 8
"""

with open(os.path.expanduser("~/.dbt/profiles.yml"), "w") as f:
    f.write(profiles_yaml.strip())

print("✅ Configured profiles.yml for databricks-cli authentication!")

In [0]:
%sh
cd globalmart_dbt
dbt debug --target dev

In [0]:
%sh
pip install "protobuf>=5.26.1,<6.0dev" "dbt-databricks" --quiet

mkdir -p globalmart_dbt/models/staging
mkdir -p globalmart_dbt/models/marts
mkdir -p globalmart_dbt/snapshots
mkdir -p globalmart_dbt/macros
mkdir -p globalmart_dbt/tests

In [0]:
import os

# Safely extract token and host using whitelisted methods
notebook_context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
token = notebook_context.apiToken().get()

try:
    host = spark.conf.get("spark.databricks.workspaceUrl")
except Exception:
    host = "dbc-73a59226-9cbc.cloud.databricks.com"

os.makedirs(os.path.expanduser("~/.dbt"), exist_ok=True)

profiles_yaml = f"""
globalmart_dbt:
  target: dev
  outputs:
    dev:
      type: databricks
      catalog: globalmart
      schema: bronze_dev
      host: "{host}"
      http_path: "/sql/1.0/warehouses/bcfd52b943980648"
      token: "{token}"
      threads: 4

    prod:
      type: databricks
      catalog: globalmart
      schema: bronze
      host: "{host}"
      http_path: "/sql/1.0/warehouses/bcfd52b943980648"
      token: "{token}"
      threads: 8
"""

with open(os.path.expanduser("~/.dbt/profiles.yml"), "w") as f:
    f.write(profiles_yaml.strip())

print("✅ profiles.yml created successfully!")

In [0]:
dbt_project_yaml = """
name: 'globalmart_dbt'
version: '1.0.0'
config-version: 2
profile: 'globalmart_dbt'

model-paths: ["models"]
analysis-paths: ["analyses"]
test-paths: ["tests"]
seed-paths: ["seeds"]
macro-paths: ["macros"]
snapshot-paths: ["snapshots"]

target-path: "target"
clean-targets: ["target", "dbt_packages"]

models:
  globalmart_dbt:
    staging:
      +materialized: view
    marts:
      +materialized: table
"""

src_bronze_yaml = """
version: 2

sources:
  - name: bronze
    database: globalmart
    schema: bronze
    freshness:
      warn_after: {count: 24, period: hour}
      error_after: {count: 48, period: hour}
    loaded_at_field: _ingested_at

    tables:
      - name: bronze_orders
        description: "Raw ingested orders dataset in Bronze layer"
        columns:
          - name: order_id
          - name: customer_id

      - name: orders_partitioned
        description: "Partitioned orders dataset"
        loaded_at_field: _ingested_at
"""

with open("globalmart_dbt/dbt_project.yml", "w") as f:
    f.write(dbt_project_yaml.strip())

with open("globalmart_dbt/models/staging/src_bronze.yml", "w") as f:
    f.write(src_bronze_yaml.strip())

print("✅ dbt_project.yml and src_bronze.yml created!")

In [0]:
%sh
cd globalmart_dbt
dbt debug --target dev
dbt debug --target prod
dbt source freshness

In [0]:
stg_orders = """
with raw_orders as (
    select * from {{ source('bronze', 'bronze_orders') }}
)
select
    order_id,
    customer_id,
    order_status,
    cast(order_purchase_timestamp as timestamp) as order_purchase_timestamp,
    cast(order_approved_at as timestamp) as order_approved_at,
    cast(order_delivered_carrier_date as timestamp) as order_delivered_carrier_date,
    cast(order_delivered_customer_date as timestamp) as order_delivered_customer_date,
    cast(order_estimated_delivery_date as timestamp) as order_estimated_delivery_date,
    order_priority,
    fulfillment_channel,
    _ingested_at
from raw_orders
where order_id is not null
"""

stg_customers = """
with raw_customers as (
    select * from {{ source('bronze', 'bronze_orders') }}
)
select distinct
    customer_id,
    order_priority,
    fulfillment_channel
from raw_customers
where customer_id is not null
"""

fct_monthly_orders = """
with orders as (
    select * from {{ ref('stg_orders') }}
)
select
    date_trunc('month', order_purchase_timestamp) as order_month,
    count(distinct order_id) as total_orders,
    count(distinct customer_id) as unique_customers,
    count(case when order_status = 'delivered' then 1 end) as delivered_orders
from orders
group by 1
"""

dim_customer_summary = """
with orders as (
    select * from {{ ref('stg_orders') }}
)
select
    customer_id,
    count(distinct order_id) as total_orders_placed,
    min(order_purchase_timestamp) as first_order_timestamp,
    max(order_purchase_timestamp) as most_recent_order_timestamp
from orders
group by 1
"""

schema_yaml = """
version: 2

models:
  - name: stg_orders
    description: "Cleaned and typed orders staging dataset from Bronze layer."
    columns:
      - name: order_id
        description: "Primary key for each order."
      - name: customer_id
        description: "Foreign key referencing customer."

  - name: stg_customers
    description: "Distinct customer records staging dataset."
    columns:
      - name: customer_id
        description: "Unique identifier for customer."

  - name: fct_monthly_orders
    description: "Gold mart table aggregating monthly order performance metrics."
    columns:
      - name: order_month
        description: "Truncated start date of order purchase month."
      - name: total_orders
        description: "Total count of orders placed in the month."

  - name: dim_customer_summary
    description: "Gold mart table capturing customer purchase activity history."
    columns:
      - name: customer_id
        description: "Unique identifier for customer."
      - name: total_orders_placed
        description: "Lifetime order count placed by customer."
"""

with open("globalmart_dbt/models/staging/stg_orders.sql", "w") as f: f.write(stg_orders.strip())
with open("globalmart_dbt/models/staging/stg_customers.sql", "w") as f: f.write(stg_customers.strip())
with open("globalmart_dbt/models/marts/fct_monthly_orders.sql", "w") as f: f.write(fct_monthly_orders.strip())
with open("globalmart_dbt/models/marts/dim_customer_summary.sql", "w") as f: f.write(dim_customer_summary.strip())
with open("globalmart_dbt/models/schema.yml", "w") as f: f.write(schema_yaml.strip())

print("✅ Models and schema.yml created!")

In [0]:
%sh
cd globalmart_dbt
dbt run
dbt docs generate

In [0]:
fct_orders_incremental = """
{{
    config(
        materialized='incremental',
        unique_key='order_id',
        incremental_strategy='merge'
    )
}}

with source_orders as (
    select * from {{ ref('stg_orders') }}
)

select
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    order_priority,
    fulfillment_channel,
    _ingested_at
from source_orders

{% if is_incremental() %}
  where _ingested_at >= (select max(_ingested_at) - interval 3 days from {{ this }})
{% endif %}
"""

with open("globalmart_dbt/models/marts/fct_orders_incremental.sql", "w") as f:
    f.write(fct_orders_incremental.strip())

print("✅ fct_orders_incremental.sql created!")

In [0]:
%sh
cd globalmart_dbt

# Initial Full Refresh Run
dbt run --select fct_orders_incremental --full-refresh

# Incremental Run (re-run on existing data to demonstrate no duplication)
dbt run --select fct_orders_incremental

In [0]:
snap_customers = """
{% snapshot snap_customers %}

{{
    config(
      target_catalog='globalmart',
      target_schema='bronze',
      unique_key='customer_id',
      strategy='timestamp',
      updated_at='order_purchase_timestamp',
    )
}}

select distinct
    customer_id,
    order_priority,
    fulfillment_channel,
    order_purchase_timestamp
from {{ ref('stg_orders') }}

{% endsnapshot %}
"""

with open("globalmart_dbt/snapshots/snap_customers.sql", "w") as f:
    f.write(snap_customers.strip())

print("✅ snap_customers.sql created!")

In [0]:
%sh
cd globalmart_dbt
dbt snapshot

In [0]:
convert_tz_macro = """
{% macro convert_tz(column_name, target_tz='Asia/Kolkata') %}
    from_utc_timestamp({{ column_name }}, '{{ target_tz }}')
{% endmacro %}
"""

singular_test = """
select
    order_month,
    total_orders
from {{ ref('fct_monthly_orders') }}
where total_orders <= 0
"""

full_schema_yaml = """
version: 2

models:
  - name: stg_orders
    columns:
      - name: order_id
        tests:
          - not_null
          - unique

  - name: fct_orders_incremental
    columns:
      - name: order_id
        tests:
          - not_null
          - unique
      - name: customer_id
        tests:
          - not_null
"""

with open("globalmart_dbt/macros/convert_tz.sql", "w") as f: f.write(convert_tz_macro.strip())
with open("globalmart_dbt/tests/assert_positive_total_orders.sql", "w") as f: f.write(singular_test.strip())
with open("globalmart_dbt/models/schema.yml", "w") as f: f.write(full_schema_yaml.strip())

print("✅ Macro, tests, and updated schema.yml created!")

In [0]:
stg_orders_macro = """
with raw_orders as (
    select * from {{ source('bronze', 'bronze_orders') }}
)
select
    order_id,
    customer_id,
    order_status,
    {{ convert_tz('order_purchase_timestamp') }} as order_purchase_timestamp_ist,
    _ingested_at
from raw_orders
where order_id is not null
"""

fct_monthly_orders_macro = """
with orders as (
    select * from {{ ref('stg_orders') }}
)
select
    date_trunc('month', order_purchase_timestamp_ist) as order_month,
    count(distinct order_id) as total_orders
from orders
group by 1
"""

with open("globalmart_dbt/models/staging/stg_orders.sql", "w") as f: f.write(stg_orders_macro.strip())
with open("globalmart_dbt/models/marts/fct_monthly_orders.sql", "w") as f: f.write(fct_monthly_orders_macro.strip())

print("✅ Applied macro across models!")

In [0]:
%sh
cd globalmart_dbt
dbt test

In [0]:
%sh
cd globalmart_dbt
dbt debug --target dev
dbt debug --target prod
dbt source freshness

In [0]:
src_bronze_yaml = """
version: 2

sources:
  - name: bronze
    database: globalmart
    schema: bronze
    loaded_at_field: _ingested_at

    tables:
      - name: bronze_orders
        description: "Raw ingested orders dataset in Bronze layer"
        freshness:
          warn_after: {count: 365, period: day}
          error_after: {count: 1000, period: day}
        columns:
          - name: order_id
          - name: customer_id

      - name: orders_partitioned
        description: "Partitioned orders dataset"
        loaded_at_field: _ingested_at
        freshness:
          warn_after: {count: 365, period: day}
          error_after: {count: 1000, period: day}
"""

with open("globalmart_dbt/models/staging/src_bronze.yml", "w") as f:
    f.write(src_bronze_yaml.strip())

print("✅ Updated src_bronze.yml freshness thresholds!")

In [0]:
%sh
cd globalmart_dbt
dbt source freshness

In [0]:
%sh
cd globalmart_dbt
dbt run
dbt docs generate

In [0]:
dim_customer_summary = """
with orders as (
    select * from {{ ref('stg_orders') }}
)
select
    customer_id,
    count(distinct order_id) as total_orders_placed,
    min(order_purchase_timestamp_ist) as first_order_timestamp,
    max(order_purchase_timestamp_ist) as most_recent_order_timestamp
from orders
group by 1
"""

fct_orders_incremental = """
{{
    config(
        materialized='incremental',
        unique_key='order_id',
        incremental_strategy='merge'
    )
}}

with source_orders as (
    select * from {{ ref('stg_orders') }}
)

select
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp_ist,
    _ingested_at
from source_orders

{% if is_incremental() %}
  where _ingested_at >= (select max(_ingested_at) - interval 3 days from {{ this }})
{% endif %}
"""

with open("globalmart_dbt/models/marts/dim_customer_summary.sql", "w") as f:
    f.write(dim_customer_summary.strip())

with open("globalmart_dbt/models/marts/fct_orders_incremental.sql", "w") as f:
    f.write(fct_orders_incremental.strip())

print("✅ Updated dim_customer_summary and fct_orders_incremental SQL!")

In [0]:
%pip install dbt-databricks

In [0]:
%sh
dbt --version

In [0]:
%sh
cd globalmart_dbt
dbt run
dbt docs generate

In [0]:
import os

# Safely extract token and workspace host
notebook_context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
token = notebook_context.apiToken().get()

try:
    host = spark.conf.get("spark.databricks.workspaceUrl")
except Exception:
    host = "dbc-73a59226-9cbc.cloud.databricks.com"

os.makedirs(os.path.expanduser("~/.dbt"), exist_ok=True)

profiles_yaml = f"""
globalmart_dbt:
  target: dev
  outputs:
    dev:
      type: databricks
      catalog: globalmart
      schema: bronze_dev
      host: "{host}"
      http_path: "/sql/1.0/warehouses/bcfd52b943980648"
      token: "{token}"
      threads: 4

    prod:
      type: databricks
      catalog: globalmart
      schema: bronze
      host: "{host}"
      http_path: "/sql/1.0/warehouses/bcfd52b943980648"
      token: "{token}"
      threads: 8
"""

with open(os.path.expanduser("~/.dbt/profiles.yml"), "w") as f:
    f.write(profiles_yaml.strip())

print("✅ Re-created ~/.dbt/profiles.yml in active environment!")

In [0]:
%sh
cd globalmart_dbt
dbt run
dbt docs generate

In [0]:
%sh
cd globalmart_dbt
dbt run --full-refresh
dbt docs generate

In [0]:
import os

# Safely extract token and workspace host
notebook_context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
token = notebook_context.apiToken().get()

try:
    host = spark.conf.get("spark.databricks.workspaceUrl")
except Exception:
    host = "dbc-73a59226-9cbc.cloud.databricks.com"

os.makedirs(os.path.expanduser("~/.dbt"), exist_ok=True)

profiles_yaml = f"""
globalmart_dbt:
  target: dev
  outputs:
    dev:
      type: databricks
      catalog: globalmart
      schema: bronze_dev
      host: "{host}"
      http_path: "/sql/1.0/warehouses/bcfd52b943980648"
      token: "{token}"
      threads: 4

    prod:
      type: databricks
      catalog: globalmart
      schema: bronze
      host: "{host}"
      http_path: "/sql/1.0/warehouses/bcfd52b943980648"
      token: "{token}"
      threads: 8
"""

with open(os.path.expanduser("~/.dbt/profiles.yml"), "w") as f:
    f.write(profiles_yaml.strip())

print("✅ Re-created ~/.dbt/profiles.yml in active environment!")

In [0]:
%sh
cd globalmart_dbt
dbt run --select fct_orders_incremental

In [0]:
%sql
SELECT 
    COUNT(*) as total_rows,
    COUNT(DISTINCT order_id) as unique_orders,
    COUNT(*) - COUNT(DISTINCT order_id) as duplicate_count
FROM globalmart.bronze_dev.fct_orders_incremental;

In [0]:
stg_orders_dedup = """
with raw_orders as (
    select * from {{ source('bronze', 'bronze_orders') }}
),
deduped as (
    select *,
        row_number() over (
            partition by order_id 
            order by _ingested_at desc, order_purchase_timestamp desc
        ) as rn
    from raw_orders
    where order_id is not null
)
select
    order_id,
    customer_id,
    order_status,
    {{ convert_tz('order_purchase_timestamp') }} as order_purchase_timestamp_ist,
    _ingested_at
from deduped
where rn = 1
"""

with open("globalmart_dbt/models/staging/stg_orders.sql", "w") as f:
    f.write(stg_orders_dedup.strip())

print("✅ stg_orders.sql updated with deduplication!")

In [0]:
%sh
cd globalmart_dbt

# Full refresh to wipe previous duplicates and apply deduplication
dbt run --select stg_orders fct_orders_incremental --full-refresh

# Re-run incrementally to prove idempotency
dbt run --select fct_orders_incremental

In [0]:
%sql
SELECT 
    COUNT(*) as total_rows,
    COUNT(DISTINCT order_id) as unique_orders,
    COUNT(*) - COUNT(DISTINCT order_id) as duplicate_count
FROM globalmart.bronze_dev.fct_orders_incremental;

In [0]:
snap_customers_sql = """
{% snapshot snap_customers %}

{{
    config(
      target_catalog='globalmart',
      target_schema='bronze_dev',
      unique_key='customer_id',
      strategy='check',
      check_cols='all'
    )
}}

select * from {{ ref('stg_customers') }}

{% endsnapshot %}
"""

with open("globalmart_dbt/snapshots/snap_customers.sql", "w") as f:
    f.write(snap_customers_sql.strip())

print("✅ Updated snap_customers.sql with check_cols='all'!")

In [0]:
stg_customers_dedup = """
with raw_customers as (
    select * from {{ source('bronze', 'bronze_orders') }}
),
deduped as (
    select *,
        row_number() over (
            partition by customer_id 
            order by _ingested_at desc
        ) as rn
    from raw_customers
    where customer_id is not null
)
select
    customer_id,
    order_priority,
    _ingested_at
from deduped
where rn = 1
"""

with open("globalmart_dbt/models/staging/stg_customers.sql", "w") as f:
    f.write(stg_customers_dedup.strip())

print("✅ stg_customers.sql updated with row_number() deduplication!")

In [0]:
%sql
DROP TABLE IF EXISTS globalmart.bronze_dev.snap_customers;

In [0]:
%sh
cd globalmart_dbt
dbt snapshot

In [0]:
%sql
SELECT 
    customer_id,
    order_priority,
    dbt_valid_from,
    dbt_valid_to,
    dbt_updated_at
FROM globalmart.bronze_dev.snap_customers
LIMIT 10;

In [0]:
import os

os.makedirs("globalmart_dbt/tests", exist_ok=True)

singular_test_sql = """
-- Custom singular test: Ensure order timestamps are not greater than ingestion timestamp
select
    order_id,
    order_purchase_timestamp_ist,
    _ingested_at
from {{ ref('stg_orders') }}
where order_purchase_timestamp_ist > _ingested_at
"""

with open("globalmart_dbt/tests/assert_order_timestamp_valid.sql", "w") as f:
    f.write(singular_test_sql.strip())

print("✅ Created custom singular test: assert_order_timestamp_valid.sql")

In [0]:
%sh
cd globalmart_dbt
dbt test